# EF5 PET/CT Tumour Growth & Hypoxia Radiotherapy Modelling Explorer

Interactive explorer for both modelling tracks on the `ef5_rt_model` branch:
1. **Sub-Goal 1**: Coupled 1D radially-symmetric quasi-steady oxygen PDE + moving-boundary spheroid RT model (extending Dr. Alex Fletcher's `CodeForBobby.m` and your 2013 `my_spheroid_07/08` ODEs across PLOS ONE single & fractionated schedules).
2. **Sub-Goal 2**: Tumour-specific 3D CT/EF5-PET image-initialised continuum voxel PDE model (3D oxygen PDE + Darcy pressure-velocity mechanics) evaluated on the longitudinal First EF5 RT Study.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

ROOT = Path('..').resolve() if Path('..', 'src').exists() else Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.physics import RadiobiologyParams
from src.modeling.spheroid_rt_model import SpheroidRTModel, make_plos_one_scenarios
from src.modeling.voxel_initializer import initialize_tumour_from_npz
from src.modeling.voxel_rt_model import VoxelRTModel, VoxelSimulationConfig

print('Workspace root:', ROOT)

## Part 1: Interactive 1D Spheroid RT Simulator (PLOS ONE Single & Fractionated Schedules)

In [ ]:
@widgets.interact(
    total_dose=widgets.FloatSlider(value=10.0, min=5.0, max=40.0, step=5.0, description='Total Gy:'),
    n_fractions=widgets.Dropdown(options=[1, 2, 4], value=1, description='Fractions:'),
    alpha=widgets.FloatSlider(value=0.20, min=0.08, max=0.32, step=0.02, description='LQ alpha:'),
    oer_max=widgets.FloatSlider(value=3.0, min=1.5, max=3.5, step=0.25, description='OER max:'),
)
def explore_spheroid(total_dose=10.0, n_fractions=1, alpha=0.20, oer_max=3.0):
    prm = RadiobiologyParams(alpha=alpha, beta=alpha / 10.0, oer_max=oer_max)
    model = SpheroidRTModel(prm)
    d_per_f = total_dose / n_fractions
    sched = [(float(i), d_per_f) for i in range(n_fractions)]
    scs = make_plos_one_scenarios(sched, f'{n_fractions}x{d_per_f:g} Gy', t_end_days=35.0)
    res = {k: model.run(sc) for k, sc in scs.items()}

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
    for k, col in [('Control', '#2ca02c'), ('EF5+', '#e41a1c'), ('EF5-', '#111111')]:
        axes[0].plot(res[k].t_days, res[k].rel_volume, color=col, lw=2.3, label=f"{k} (pre T/M={res[k].mean_ef5_tm[0]:.2f})")
        axes[1].plot(res[k].t_days, res[k].mean_po2_mmHg, color=col, lw=2.2, label=k)
        axes[2].plot(res[k].t_days, res[k].mean_ef5_tm, color=col, lw=2.2, label=k)
    axes[0].axhline(1.0, color='gray', ls=':', lw=0.9)
    axes[0].set_title(f'Normalized Volume ({n_fractions} x {d_per_f:g} Gy)')
    axes[0].set_ylabel('V / V0')
    axes[1].set_title('Mean Tumour pO2 (Reoxygenation)')
    axes[1].set_ylabel('pO2 (mmHg)')
    axes[2].set_title('Predicted Virtual EF5 T/M')
    axes[2].set_ylabel('EF5 T/M')
    for ax in axes:
        ax.set_xlabel('Days Post-RT')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

## Part 2: Interactive 3D Image-Initialised Mouse Tumour Simulator

In [ ]:
mouse_id = 'c2_M1R'  # Options: c2_M1R, c2_M0, c2_M1L, c2_M1L1R, c2_M2R
npz_m1 = ROOT / 'outputs' / 'first_ef5_rt_study' / 'voxel_bundles' / mouse_id / f'{mouse_id}_day-1_2010-12-02.npz'

prm = RadiobiologyParams()
model_3d = VoxelRTModel(prm)
st_rt = initialize_tumour_from_npz(npz_m1, 'rt', params=prm, dx_mm=0.5)
st_ctrl = initialize_tumour_from_npz(npz_m1, 'ctrl', params=prm, dx_mm=0.5)

res_rt = model_3d.run(st_rt, VoxelSimulationConfig(t_end_days=33.0, dt_days=0.5, dose_schedule=[(1.0, 20.0)], snapshot_days=[0.0, 14.0, 33.0]))
res_ctrl = model_3d.run(st_ctrl, VoxelSimulationConfig(t_end_days=33.0, dt_days=0.5, dose_schedule=[], snapshot_days=[0.0, 14.0, 33.0]))

print(f"{mouse_id} RT:   Vol {res_rt.volume_cm3[0]:.3f} -> {res_rt.volume_cm3[-1]:.3f} cm^3 | T/M {res_rt.mean_ef5_tm[0]:.2f} -> {res_rt.mean_ef5_tm[-1]:.2f}")
print(f"{mouse_id} Ctrl: Vol {res_ctrl.volume_cm3[0]:.3f} -> {res_ctrl.volume_cm3[-1]:.3f} cm^3 | T/M {res_ctrl.mean_ef5_tm[0]:.2f} -> {res_ctrl.mean_ef5_tm[-1]:.2f}")

In [ ]:
@widgets.interact(
    side=widgets.Dropdown(options=['RT (20 Gy)', 'Control (0 Gy)'], value='RT (20 Gy)'),
    snap_day=widgets.Dropdown(options=[('Day -1 (Pre-RT)', 0.0), ('Day +13', 14.0), ('Day +32', 33.0)], value=0.0),
    z_idx=widgets.IntSlider(value=24, min=0, max=48, step=1, description='Coronal Z:'),
)
def view_3d_slices(side='RT (20 Gy)', snap_day=0.0, z_idx=24):
    res = res_rt if side.startswith('RT') else res_ctrl
    snap = res.snapshots[snap_day]
    zi = min(z_idx, snap['ef5_tm'].shape[1] - 1)
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
    im0 = axes[0].imshow(snap['ef5_tm'][:, zi, :], cmap='inferno', vmin=0.8, vmax=4.8, origin='lower')
    axes[0].contour(snap['roi_mask'][:, zi, :].astype(float), levels=[0.5], colors='cyan', linewidths=1.5)
    axes[0].set_title('Simulated EF5 PET T/M')
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(snap['p_o2'][:, zi, :], cmap='viridis', vmin=0.0, vmax=35.0, origin='lower')
    axes[1].contour(snap['roi_mask'][:, zi, :].astype(float), levels=[0.5], colors='white', linewidths=1.5)
    axes[1].set_title('Simulated pO2 (mmHg)')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(snap['V'][:, zi, :], cmap='magma', vmin=0.0, vmax=1.0, origin='lower')
    axes[2].set_title('Viable Cell Density V(x)')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)
    fig.tight_layout()
    plt.show()